# FinSim — Session 01
## Simülasyon Motoru: Matematiksel Temel

**Tarih:** 4 Temmuz 2026  
**Kapsam:** Enflasyon, döviz, altın, BIST, mevduat modellerinin matematiksel temeli ve entegrasyonu.  
**Sonraki session:** Gayrimenkul modeli, entegrasyon testi genişletme, JS'e taşıma.

---

## 0. Yardımcı Fonksiyonlar

In [ ]:
import random
import math

def normal(ortalama, std, min_val=None, max_val=None):
    """
    Normal dağılımdan rastgele sayı üretir.
    random.gauss() kullanır — Box-Muller'dan daha güvenilir.
    min_val / max_val ile değer aralığı kısıtlanabilir.
    """
    deger = random.gauss(ortalama, std)
    if min_val is not None:
        deger = max(deger, min_val)
    if max_val is not None:
        deger = min(deger, max_val)
    return round(deger, 4)

---
## 1. Enflasyon Motoru

### Yaklaşım: İki Rejimli Markov Sistemi

Türkiye enflasyonu iki modda seyrediyor:
- **Sakin rejim:** %6–18 arası, düşük oynaklık  
- **Kriz rejimi:** %50–80 arası başlayıp yavaş iniyor

Tarihsel veri (TÜİK / World Bank):
```
2001: %54.4  |  2002: %45.0  |  2003: %25.3  |  2004: %10.6
2009: %6.3   |  2018: %16.3  |  2021: %19.6  |  2022: %72.3
2023: %53.9  |  2024: %58.5  |  2025: %34.9
```

### Kritik Kararlar

**Exponansiyel kriz birikimi:** Sabit olasılık yerine her sakin yılda gerilim artar.  
Formül: `P(kriz) = 1 - e^(−0.04 × sakin_yıl)` · İlk 3 yılda kriz çıkmaz.

**Randomize kriz parametreleri:** Her krizin başlangıç seviyesi ve düşüş hızı ayrı üretilir → her oyun farklı hissettiriyor.

Türetilen parametreler:
```
Sakin:    ort. %9.5,  std %2.8,  aralık [%5, %18]
Kriz:     ort. %63,   std %8,    aralık [%50, %80]
Düşüş:   ort. %27/yıl, std %5,  aralık [%18, %38]
Eşik:    %18 altına düşünce sakin rejime dönüş
```

In [ ]:
def enflasyon_sim(rejim, sakin_yil, kriz_mevcut, kriz_dusus_hizi):
    """
    İki rejimli Markov enflasyon modeli.
    rejim: 0=sakin, 1=kriz
    Döndürür: (enflasyon, rejim, sakin_yil, kriz_mevcut, kriz_dusus_hizi, durum)
    """
    if rejim == 0:  # SAKİN
        kriz_olasiligi = 1 - math.exp(-0.04 * sakin_yil)
        if random.random() < kriz_olasiligi and sakin_yil >= 3:
            rejim = 1
            sakin_yil = 0
            kriz_mevcut = normal(63, 8, min_val=50, max_val=80)
            kriz_dusus_hizi = normal(27, 5, min_val=18, max_val=38)
            enf = normal(kriz_mevcut, 4, min_val=kriz_mevcut*0.9, max_val=kriz_mevcut*1.1)
            durum = "KRİZ"
        else:
            sakin_yil += 1
            enf = normal(9.5, 2.8, min_val=5, max_val=18)
            durum = "sakin"
    else:  # KRİZ
        gercek_dusus = normal(kriz_dusus_hizi, 5, min_val=18, max_val=45)
        kriz_mevcut = kriz_mevcut * (1 - gercek_dusus / 100)
        if kriz_mevcut <= 18:
            rejim = 0
            sakin_yil = 0
            enf = normal(14, 3, min_val=8, max_val=20)
            durum = "normalleşme"
            kriz_mevcut = None
        else:
            enf = normal(kriz_mevcut, 6,
                        min_val=kriz_mevcut*0.85, max_val=kriz_mevcut*1.15)
            durum = "kriz devam"

    return enf, rejim, sakin_yil, kriz_mevcut, kriz_dusus_hizi, durum

In [ ]:
# Test — 3 farklı seed ile 60 yıllık simülasyon
for i, seed in enumerate([42, 99, 777]):
    random.seed(seed)
    rejim, sakin_yil, kriz_mevcut, kriz_dusus = 0, 0, None, None
    sonuclar = []
    for yil in range(60):
        enf, rejim, sakin_yil, kriz_mevcut, kriz_dusus, durum = \
            enflasyon_sim(rejim, sakin_yil, kriz_mevcut, kriz_dusus)
        sonuclar.append({"yil": 2025+yil, "yas": 25+yil,
                         "enflasyon": round(enf,1), "rejim": rejim, "durum": durum})

    kriz_sayisi = sum(1 for s in sonuclar if s["durum"] == "KRİZ")
    enf_values = [s["enflasyon"] for s in sonuclar]
    print(f"\n=== Sim {i+1} (seed={seed}) ===")
    print(f"Kriz: {kriz_sayisi} · Ort. enf: %{round(sum(enf_values)/60,1)} · "
          f"Min: %{min(enf_values)} · Max: %{max(enf_values)}")

---
## 2. Döviz Modeli (USD/TRY)

### Yaklaşım: Enflasyona Bağlı Çarpan

```
döviz_değişim = enflasyon × rassal_çarpan
```

Tarihsel çarpan verileri:
```
2018: %40.1 döviz / %16.3 enf → çarpan 2.46
2021: %79.0 döviz / %19.6 enf → çarpan 4.03  (şok yılı)
2022: %40.5 döviz / %72.3 enf → çarpan 0.56  (Şimşek dönemi)
2024: %14.9 döviz / %58.5 enf → çarpan 0.25  (TL güçlendi)
```

### Kur Patlaması ve Redenominasyon

**Problem:** 60 yılda kur astronomik rakamlara ulaşıyor.  
**Çözüm:** Kur ≥ 1000 TL olunca "Yeni TL" eventi tetiklenir, tüm değerler 1000'e bölünür.  
**Davranışsal değeri:** Kullanıcı 1.200.000 TL'nin birdenbire 1.200 TL'ye dönüştüğünü görür —  
nominal değişime rağmen satın alma gücü aynı. Açıklama yapmadan hissettiriyor.

**Gelecek görev:** Oyun sonu raporunda redenominasyon dönemlerini kapsayan reel endeks.

In [ ]:
def doviz_sim(enflasyon, rejim):
    """
    Döviz değer kaybı = enflasyon × rassal çarpan.
    Sakin: N(2.0, 0.6) aralık [0.8, 4.0]
    Kriz:  N(1.5, 0.8) aralık [0.5, 5.0]
    """
    if rejim == 0:
        carpan = normal(2.0, 0.6, min_val=0.8, max_val=4.0)
    else:
        carpan = normal(1.5, 0.8, min_val=0.5, max_val=5.0)
    return round(enflasyon * carpan, 1), round(carpan, 2)

---
## 3. Altın Fiyat Modeli (USD Bazında)

### Neden GBM Değil?

GBM her yılın bağımsız olduğunu varsayar. Altın ise momentum ve rejim bazlı hareket eder:
```
1971-1980  Boğa: ~9 yıl,  ort. +%35/yıl
1980-1999  Ayı:  ~19 yıl, ort. -%4/yıl
1999-2011  Boğa: ~12 yıl, ort. +%15/yıl
2011-2018  Ayı:  ~7 yıl,  ort. -%5/yıl
2018-2026  Boğa: ~8 yıl+  ort. +%18/yıl
```

### Üç Rejim Parametreleri
```
Durgun: ort. +%2,  std %8,   aralık [-%5, +%12]  · 5-8 yıl tipik
Boğa:   ort. +%20, std %18,  aralık [+%5, +%50]  · 8-12 yıl tipik
Ayı:    ort. -%12, std %12,  aralık [-%35, -0.001] · 3-5 yıl tipik
```

### Kritik Kararlar
- Ayıda pozitif getiri yok: `max = -0.001`
- Zirve fiyatı ayı başlayınca sabitleniyor — "tepeden ne kadar düştüm" doğru hesaplanıyor
- Enflasyon kriz rejimindeyken boğa olasılığı +%15 artıyor

In [ ]:
ALTIN_REJIMLER = {
    "durgun": {"getiri_ort": 0.02, "getiri_std": 0.08,
               "getiri_min": -0.05, "getiri_max": 0.12},
    "boga":   {"getiri_ort": 0.20, "getiri_std": 0.18,
               "getiri_min": 0.05,  "getiri_max": 0.50},
    "ayi":    {"getiri_ort": -0.12, "getiri_std": 0.12,
               "getiri_min": -0.35, "getiri_max": -0.001},
}
AYI_ESIGI = -0.35


def altin_sim(rejim, durgun_yil, boga_yil, zirve_fiyat, fiyat, enflasyon_rejim=0):
    """
    Üç rejimli altın USD fiyat modeli.
    rejim: 0=durgun, 1=boğa, 2=ayı
    """
    if rejim == 0:  # DURGUN
        boga_olasiligi = 1 - math.exp(-0.07 * durgun_yil)
        if enflasyon_rejim == 1:
            boga_olasiligi = min(boga_olasiligi + 0.15, 0.90)
        if random.random() < boga_olasiligi and durgun_yil >= 2:
            rejim, durgun_yil, boga_yil = 1, 0, 1
            zirve_fiyat = fiyat
            p = ALTIN_REJIMLER["boga"]
            getiri = normal(p["getiri_ort"], p["getiri_std"],
                           min_val=p["getiri_min"], max_val=p["getiri_max"])
            durum = "BOĞA BAŞLADI"
        else:
            durgun_yil += 1
            p = ALTIN_REJIMLER["durgun"]
            getiri = normal(p["getiri_ort"], p["getiri_std"],
                           min_val=p["getiri_min"], max_val=p["getiri_max"])
            durum = f"durgun ({durgun_yil}. yıl)"

    elif rejim == 1:  # BOĞA
        boga_yil += 1
        ayi_olasiligi = 0.05 if boga_yil <= 8 else 0.05 + (boga_yil - 8) * 0.12
        if random.random() < ayi_olasiligi:
            rejim, boga_yil = 2, 0
            zirve_fiyat = fiyat
            p = ALTIN_REJIMLER["ayi"]
            getiri = normal(p["getiri_ort"], p["getiri_std"],
                           min_val=p["getiri_min"], max_val=p["getiri_max"])
            durum = "AYI BAŞLADI"
        else:
            p = ALTIN_REJIMLER["boga"]
            getiri = normal(p["getiri_ort"], p["getiri_std"],
                           min_val=p["getiri_min"], max_val=p["getiri_max"])
            durum = f"boğa ({boga_yil}. yıl)"

    else:  # AYI
        p = ALTIN_REJIMLER["ayi"]
        getiri = normal(p["getiri_ort"], p["getiri_std"],
                       min_val=p["getiri_min"], max_val=p["getiri_max"])
        tepe_dusus = (fiyat - zirve_fiyat) / zirve_fiyat
        if tepe_dusus <= AYI_ESIGI:
            rejim, durgun_yil = 0, 0
            durum = "durgun (ayıdan çıkış)"
        else:
            durum = f"ayı (tepeden %{round(tepe_dusus*100,1)})"

    yeni_fiyat = round(fiyat * (1 + getiri), 2)
    if rejim == 1:
        zirve_fiyat = max(zirve_fiyat, yeni_fiyat)

    return getiri, rejim, durgun_yil, boga_yil, zirve_fiyat, yeni_fiyat, durum

---
## 4. BIST ve Faiz Modeli

### Neden Farklı?

BIST uzun rejim dönemlerine sahip değil — yıllık getiriler çok daha kısa ve sert dönüşler yapıyor.  
**Model:** Politika faizi yönü ana eğilimi belirliyor + rastgele şok.

### Politika Faizi
- Enflasyona bağlı çarpanla, kademeli hareket (yılda max %30 sakin, %50 kriz)
- Sapma eventi (%8 olasılık): 2021 Türkiye senaryosu — enflasyon yükselirken faiz düşürülüyor

### BIST Getiri Modeli
```
BIST = faiz_yön_biası + enflasyondan_kaçış + rastgele_şok

Faiz ↓: bias = N(+0.40, 0.15)  aralık [0.0, +0.90]
Faiz ↑: bias = N(+0.15, 0.15)  aralık [-0.30, +0.50]
Sabit:  bias = N(+0.25, 0.12)  aralık [-0.10, +0.50]
Kriz kaçış etkisi: +N(0.15, 0.10) aralık [0.0, +0.35]
Şok: N(0, 0.20) aralık [-0.40, +0.40]
```

Tarihsel kalibrasyon (2000-2024): Ort. TRY ~%30/yıl · Pozitif reel ~%52 yıl

In [ ]:
def faiz_uret(enflasyon, rejim, onceki_faiz, sapma_olasiligi=0.08):
    """
    Politika faizini kademeli olarak üretir.
    Sapma eventi: faiz enflasyonun çok altında (politik baskı).
    """
    sapma = random.random() < sapma_olasiligi
    if sapma:
        hedef = enflasyon * normal(0.2, 0.08, min_val=0.1, max_val=0.4)
        mod = "SAPMA"
    elif rejim == 0:
        hedef = enflasyon * normal(0.95, 0.10, min_val=0.6, max_val=1.3)
        mod = "normal"
    else:
        hedef = enflasyon * normal(0.75, 0.15, min_val=0.45, max_val=1.1)
        mod = "kriz"

    max_degisim = onceki_faiz * (0.50 if rejim == 1 else 0.30)
    faiz = max(onceki_faiz - max_degisim,
               min(onceki_faiz + max_degisim, hedef))
    faiz = round(max(faiz, 2.0), 1)

    if faiz > onceki_faiz * 1.05:   yon = "↑ artıyor"
    elif faiz < onceki_faiz * 0.95: yon = "↓ düşüyor"
    else:                            yon = "→ sabit"

    return faiz, yon, mod


def bist_getiri_uret(faiz_yon, enflasyon, enf_rejim=0):
    """
    BIST TRY getirisi = faiz yön biası + kaçış etkisi + rastgele şok.
    """
    if "↓" in faiz_yon:
        bias = normal(0.40, 0.15, min_val=0.0, max_val=0.90)
    elif "↑" in faiz_yon:
        bias = normal(0.15, 0.15, min_val=-0.30, max_val=0.50)
    else:
        bias = normal(0.25, 0.12, min_val=-0.10, max_val=0.50)

    if enf_rejim == 1:  # kriz döneminde borsaya kaçış
        bias += normal(0.15, 0.10, min_val=0.0, max_val=0.35)

    sok = normal(0, 0.20, min_val=-0.40, max_val=0.40)
    getiri = (bias + sok) * 100
    return max(-55.0, min(100.0, getiri))

---
## 5. Mevduat Modeli

Vadeli mevduat doğrudan enflasyona bağlı çarpanla modelleniyor.

```
Sakin: mevduat = enflasyon × N(0.95, 0.12)  aralık [0.70, 1.20]
Kriz:  mevduat = enflasyon × N(0.65, 0.15)  aralık [0.40, 0.90]
```

Kriz döneminde mevduat enflasyonun gerisinde kalır — **vadeli mevduat illüzyonu burada.**

In [ ]:
def mevduat_faizi_uret(enflasyon, rejim):
    """
    Vadeli mevduat faizi enflasyona bağlı çarpanla üretiliyor.
    Döndürür: (faiz, carpan, reel_getiri)
    """
    if rejim == 0:
        carpan = normal(0.95, 0.12, min_val=0.70, max_val=1.20)
    else:
        carpan = normal(0.65, 0.15, min_val=0.40, max_val=0.90)

    faiz = round(enflasyon * carpan, 1)
    reel = round(faiz - enflasyon, 1)
    return faiz, round(carpan, 2), reel

---
## 6. Gayrimenkul (Placeholder)

Gayrimenkul başlı başına bir özellik olarak Session 02/03'te detaylandırılacak.  
Şimdilik enflasyonla orantılı basit bir değer artışı kullanılıyor.

In [ ]:
def gayrimenkul_getiri(enflasyon, rejim):
    """PLACEHOLDER — Session 02/03'te gerçek modelle değiştirilecek."""
    carpan = normal(1.1, 0.20, min_val=0.8, max_val=1.5)
    return round(enflasyon * carpan, 1)

---
## 7. Tam Entegrasyon — simulate_finsim()

Tüm modeller tek fonksiyonda birleşiyor.

**Veri akışı:**
```
Enflasyon (merkezi parametre)
    ├── Döviz (enflasyon × çarpan)
    ├── Altın TRY (altın USD × döviz etkisi)
    ├── BIST (faiz yönü → bias + şok)
    ├── Mevduat (enflasyon × çarpan)
    └── Gayrimenkul (placeholder)
```

In [ ]:
def simulate_finsim(yil_sayisi=60, seed=None):
    """Ana simülasyon fonksiyonu. Tüm varlık modellerini entegre eder."""
    if seed:
        random.seed(seed)

    # State
    enf_rejim, enf_sakin_yil, enf_kriz_mevcut, enf_kriz_dusus = 0, 0, None, None
    kur, kur_ham, redenominasyon_sayaci, KUR_ESIK = 40.0, 40.0, 0, 1000
    altin_usd, altin_zirve = 2600.0, 2600.0
    altin_rejim, altin_durgun_yil, altin_boga_yil = 1, 0, 7
    faiz, bist = 12.0, 100.0
    mevduat_birikim, gayrimenkul = 100.0, 100.0

    sonuclar = []

    for yil in range(yil_sayisi):

        # 1. ENFLASYON
        enf, enf_rejim, enf_sakin_yil, enf_kriz_mevcut, enf_kriz_dusus, enf_durum = \
            enflasyon_sim(enf_rejim, enf_sakin_yil, enf_kriz_mevcut, enf_kriz_dusus)

        # 2. DÖVİZ
        doviz_degisim, doviz_carpan = doviz_sim(enf, enf_rejim)
        kur_ham = round(kur_ham * (1 + doviz_degisim / 100), 2)
        kur = round(kur * (1 + doviz_degisim / 100), 2)
        redenominasyon = None
        if kur >= KUR_ESIK:
            kur = round(kur / 1000, 2)
            redenominasyon_sayaci += 1
            redenominasyon = f"YENİ TL #{redenominasyon_sayaci}"

        # 3. ALTIN
        altin_getiri, altin_rejim, altin_durgun_yil, altin_boga_yil, \
        altin_zirve, altin_usd, altin_durum = \
            altin_sim(altin_rejim, altin_durgun_yil, altin_boga_yil,
                     altin_zirve, altin_usd, enf_rejim)
        altin_try_getiri = round(altin_getiri * 100 + doviz_degisim, 1)

        # 4. BIST
        faiz, faiz_yon, faiz_mod = faiz_uret(enf, enf_rejim, faiz)
        bist_pct = bist_getiri_uret(faiz_yon, enf, enf_rejim)
        bist = round(bist * (1 + bist_pct / 100), 2)

        # 5. MEVDUAT
        mev_faiz, mev_carpan, mev_reel = mevduat_faizi_uret(enf, enf_rejim)
        mevduat_birikim = round(mevduat_birikim * (1 + mev_faiz / 100), 2)

        # 6. GAYRİMENKUL (placeholder)
        gm_getiri = gayrimenkul_getiri(enf, enf_rejim)
        gayrimenkul = round(gayrimenkul * (1 + gm_getiri / 100), 2)

        sonuclar.append({
            "yil": 2025 + yil, "yas": 25 + yil,
            "enflasyon": round(enf, 1), "enf_rejim": enf_rejim, "enf_durum": enf_durum,
            "faiz": faiz, "faiz_yon": faiz_yon,
            "kur": kur, "kur_ham": kur_ham,
            "doviz_degisim": round(doviz_degisim, 1),
            "reel_doviz": round(doviz_degisim - enf, 1),
            "redenominasyon": redenominasyon,
            "altin_usd": altin_usd,
            "altin_try_getiri": altin_try_getiri,
            "reel_altin": round(altin_try_getiri - enf, 1),
            "altin_durum": altin_durum,
            "bist": bist, "bist_pct": round(bist_pct, 1),
            "reel_bist": round(bist_pct - enf, 1),
            "mev_faiz": mev_faiz,
            "mevduat_birikim": mevduat_birikim,
            "reel_mevduat": mev_reel,
            "gayrimenkul": gayrimenkul, "gm_getiri": gm_getiri,
            "reel_gm": round(gm_getiri - enf, 1),
        })

    return sonuclar

In [ ]:
# Entegrasyon Testi
print("=== FİNSİM — TAM ENTEGRASYON TESTİ ===\n")

for i, seed in enumerate([42, 99, 777]):
    sim = simulate_finsim(60, seed)

    enf_ort   = round(sum(s["enflasyon"] for s in sim) / 60, 1)
    bist_ort  = round(sum(s["bist_pct"] for s in sim) / 60, 1)
    altin_ort = round(sum(s["altin_try_getiri"] for s in sim) / 60, 1)
    doviz_ort = round(sum(s["doviz_degisim"] for s in sim) / 60, 1)
    mev_ort   = round(sum(s["mev_faiz"] for s in sim) / 60, 1)

    reel_bist  = sum(1 for s in sim if s["reel_bist"] > 0)
    reel_altin = sum(1 for s in sim if s["reel_altin"] > 0)
    reel_mev   = sum(1 for s in sim if s["reel_mevduat"] > 0)
    reden      = sum(1 for s in sim if s["redenominasyon"])
    kriz       = sum(1 for s in sim if s["enf_rejim"] == 1)

    print(f"\n{'='*60}")
    print(f"Simülasyon {i+1} (seed={seed})")
    print(f"{'='*60}")
    print(f"Kriz yılları: {kriz}/60 · Redenominasyon: {reden} kez · Ort. enf: %{enf_ort}")
    print()
    print(f"{'Varlık':<14} {'Ort. TRY%':<12} {'Reel>0'}")
    print("-" * 35)
    print(f"{'BIST':<14} %{bist_ort:<11} {reel_bist}/60")
    print(f"{'Altın (TRY)':<14} %{altin_ort:<11} {reel_altin}/60")
    print(f"{'Döviz':<14} %{doviz_ort:<11} -")
    print(f"{'Mevduat':<14} %{mev_ort:<11} {reel_mev}/60")

    print(f"\n{'Yıl':<6}{'Yaş':<5}{'Enfl.':<8}{'BIST%':<9}{'Altn%':<9}"
          f"{'Dvz%':<8}{'Mev%':<8}{'Kur':<10}Durum")
    print("-" * 75)
    for s in sim:
        icon = "🔴" if s["enf_rejim"] == 1 else "🟢"
        rd   = " ⚡" if s["redenominasyon"] else ""
        b = f"+%{round(s['bist_pct'],1)}" if s['bist_pct'] >= 0 else f"%{round(s['bist_pct'],1)}"
        a = f"+%{s['altin_try_getiri']}" if s['altin_try_getiri'] >= 0 else f"%{s['altin_try_getiri']}"
        d = f"+%{s['doviz_degisim']}" if s['doviz_degisim'] >= 0 else f"%{s['doviz_degisim']}"
        print(f"{s['yil']:<6}{s['yas']:<5}%{s['enflasyon']:<7}"
              f"{b:<9}{a:<9}{d:<8}%{s['mev_faiz']:<7}{s['kur']:<10}{icon}{rd}")

---
## Notlar ve Açık Görevler

### Bilinen Sınırlamalar
- **Kur patlaması:** Redenominasyon event'i eklendi ama oyun sonu raporunda reel endeks gerekiyor
- **Kur dengeleme eventi:** Merkez bankası müdahalesi gibi nadir eventlerle dengelenecek (backlog'da)
- **Gayrimenkul:** Placeholder — Session 02/03'te kira getirisi, konum etkisi, mortgage ile detaylandırılacak

### Session 02 Hedefleri
- [ ] Gayrimenkul modeli (gerçek)
- [ ] Kur dengeleme random event
- [ ] Tüm modellerin JavaScript'e taşınması
- [ ] Varlıklar arası korelasyon ince ayarı